# Web-Gold-40K reviewed-overlay validation

Run this CPU notebook only after `scripts/reconcile_gold_reviews.py --require-pass` succeeds. Attach the existing `kiyasmahmud/web-gold-40k` dataset and the passed reconciliation output. The notebook applies approved decisions in memory to train/validation, reads zero test rows, and never writes a replacement dataset.

In [ ]:
# 1. Pull the reviewed implementation.
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/Kiyas-Mahmud/webagent.git'
REPO_ROOT = Path('/kaggle/working/webagent')
if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'Code'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'Code', '--single-branch', REPOSITORY, str(REPO_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(REPO_ROOT)], check=True)
print('Repository:', REPO_ROOT)

In [ ]:
# 2. Locate exactly one passed reconciliation and the mounted source dataset.
INPUT_ROOT = Path('/kaggle/input')
DATA_ROOT = Path('/kaggle/input/datasets/kiyasmahmud/web-gold-40k')
if not DATA_ROOT.is_dir():
    DATA_ROOT = INPUT_ROOT

matches = sorted(INPUT_ROOT.rglob('review_reconciliation_report.json'))
if len(matches) != 1:
    raise RuntimeError(
        'Attach exactly one passed reconciliation output; '
        f'found reports: {matches}'
    )
RECONCILIATION_DIR = matches[0].parent
REPORT_PATH = Path('/kaggle/working/gold_review_overlay_validation.json')
print('Dataset search root:', DATA_ROOT)
print('Reconciliation:', RECONCILIATION_DIR)

In [ ]:
# 3. Validate the overlay against train/validation only.
import json

command = [
    sys.executable,
    str(REPO_ROOT / 'scripts' / 'validate_gold_review_overlay.py'),
    '--data-root', str(DATA_ROOT),
    '--reconciliation-dir', str(RECONCILIATION_DIR),
    '--report', str(REPORT_PATH),
]
print('Running:', ' '.join(command))
result = subprocess.run(command, check=False)
assert result.returncode == 0, f'Overlay validator exited {result.returncode}'
report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
assert report['status'] == 'PASS'
assert report['controlled_mini_permitted'] is True
assert report['publication_ready'] is False
assert report['test_rows_read'] == 0
assert report['source_records_mutated'] is False
print('REVIEW OVERLAY VALIDATION PASSED')
print('Report:', REPORT_PATH)
print('Use this reconciliation directory as data.review_overlay_dir in the next controlled mini:')
print(RECONCILIATION_DIR)

A pass permits the controlled 5k mini only. It does not mark the full corpus publication-ready, and it does not authorize reading the test split.